Extract competencies and qualifications

In [2]:
%run vectorizing_Options1.ipynb import df

5935 lines retrieved from the database
best_k = 42, score = 0.9991575400168492
Cluster: 9 | Category: Commercial / Vente | Number of titles: 5096
0              conseiller client
1    administration comptabilite
2             procurment officer
3                  coursier moto
4         premiere seconde ligne
Name: title_clean, dtype: object
Cluster: 3 | Category: Marketing / Digital | Number of titles: 205
40       projet digital projets moa
42              projet digital site
44    projet digital chantier metal
56             projet digital depot
57    projet digital projet seo sea
Name: title_clean, dtype: object
Cluster: 2 | Category: Support / Service Client | Number of titles: 152
49                                          tele prospecteur
63     tele propsecteurphonique specialise tourisme bilingue
108                                         tele prospecteur
155                                         tele prospecteur
157                                         tele prospecteur

In [3]:
df.keys()

Index(['title_clean', 'mission_clean', 'cluster', 'famille_poste'], dtype='object')

## Define Skill Dictionaries
Comprehensive skills across all job domains

In [5]:
import re
from collections import Counter

# Define skill dictionaries for extraction - All job categories in French

# IT & Tech skills
skills_informatique = {
    'programmation': [
        'python', 'java', 'javascript', 'php', 'c#', 'csharp', 'typescript', 'ruby', 'go', 'rust',
        'kotlin', 'swift', 'r', 'matlab', 'scala', 'perl', 'dart', 'html', 'css', 'sql', 'developpement',
        'programmation', 'codage', 'developpeur', 'programmeur'
    ],
    'frameworks': [
        'django', 'flask', 'fastapi', 'spring', 'springboot', 'angular', 'react', 'vue', 'nodejs',
        'express', 'laravel', 'symfony', 'dotnet', 'asp net', 'jquery', 'bootstrap', 'tailwind'
    ],
    'database': [
        'mysql', 'postgresql', 'mongodb', 'oracle', 'sql server', 'sqlite', 'redis', 'cassandra',
        'mariadb', 'base de donnees', 'sgbd', 'requete sql'
    ],
    'outils_tech': [
        'git', 'docker', 'kubernetes', 'jenkins', 'gitlab', 'github', 'azure', 'aws', 'gcp',
        'linux', 'windows server', 'apache', 'nginx', 'terraform', 'ansible', 'ci cd', 'devops',
        'reseau', 'serveur', 'infrastructure', 'cloud', 'virtualisation'
    ],
    'data_bi': [
        'power bi', 'tableau', 'excel', 'data analysis', 'reporting', 'kpi', 'dashboard',
        'machine learning', 'ia', 'intelligence artificielle', 'etl', 'data warehouse', 'big data'
    ]
}

# Finance & Accounting skills
skills_finance_compta = {
    'comptabilite': [
        'comptabilite', 'comptable', 'bilan', 'compte resultat', 'grand livre', 'balance',
        'saisie comptable', 'ecritures comptables', 'rapprochement bancaire', 'cloture comptable',
        'declarations fiscales', 'fiscalite', 'tva', 'tresorerie', 'immobilisations'
    ],
    'finance': [
        'finance', 'financier', 'budget', 'budgetaire', 'previsionnel', 'analyse financiere',
        'controle de gestion', 'tableau de bord', 'reporting financier', 'audit', 'audit interne',
        'controle budgetaire', 'cash flow', 'flux tresorerie'
    ],
    'outils_compta': [
        'sage', 'ciel', 'erp', 'sap', 'oracle financials', 'excel avance', 'tableaux croises dynamiques'
    ]
}

# Commercial & Sales skills
skills_commercial_vente = {
    'vente': [
        'vente', 'vendeur', 'commercial', 'prospection', 'prospection commerciale', 'closing',
        'negociation commerciale', 'argumentaire', 'techniques de vente', 'vente btob', 'vente btoc',
        'teleprospection', 'relance client', 'fidelisation', 'portefeuille client'
    ],
    'relation_client': [
        'relation client', 'service client', 'satisfaction client', 'gestion reclamation',
        'accueil client', 'conseils client', 'ecoute client', 'suivi client', 'crm'
    ],
    'business_dev': [
        'business development', 'developpement commercial', 'strategie commerciale', 'etude de marche',
        'plan action commercial', 'objectifs commerciaux', 'chiffre affaires'
    ]
}

# HR & Management skills
skills_rh_management = {
    'ressources_humaines': [
        'recrutement', 'sourcing', 'entretien embauche', 'onboarding', 'formation', 'plan de formation',
        'gestion carriere', 'evaluation performance', 'gpec', 'paie', 'gestion paie', 'contrat travail',
        'droit du travail', 'relations sociales', 'administration personnel'
    ],
    'management': [
        'management', 'manager', 'encadrement', 'gestion equipe', 'leadership', 'supervision',
        'coordination equipe', 'animation equipe', 'motivation equipe', 'coaching', 'delegation',
        'prise de decision', 'gestion conflit', 'conduite reunion'
    ]
}

# Supply Chain & Logistics skills
skills_supply_logistique = {
    'logistique': [
        'logistique', 'supply chain', 'gestion stock', 'gestion entrepot', 'inventaire',
        'preparation commande', 'expedition', 'reception marchandise', 'manutention',
        'cariste', 'chariot elevateur', 'caces', 'stockage', 'picking'
    ],
    'transport': [
        'transport', 'livraison', 'acheminement', 'planning livraison', 'tournee',
        'permis poids lourd', 'permis b', 'conduite', 'chauffeur'
    ],
    'approvisionnement': [
        'approvisionnement', 'achat', 'procurement', 'fournisseur', 'negociation fournisseur',
        'commande', 'suivi commande', 'import export', 'douane', 'incoterms'
    ]
}

# Marketing & Communication skills
skills_marketing_communication = {
    'marketing': [
        'marketing', 'marketing digital', 'strategie marketing', 'plan marketing', 'positionnement',
        'etude marche', 'segmentation', 'ciblage', 'brand management', 'marketing produit'
    ],
    'digital': [
        'seo', 'sem', 'referencement', 'google ads', 'social media', 'reseaux sociaux', 'facebook ads',
        'community management', 'content marketing', 'inbound marketing', 'email marketing',
        'marketing automation', 'web analytics', 'google analytics'
    ],
    'communication': [
        'communication', 'redaction', 'redaction contenu', 'storytelling', 'relations publiques',
        'relations presse', 'evenementiel', 'communication interne', 'communication externe'
    ]
}

# Industrial & Technical skills
skills_industriel_technique = {
    'production': [
        'production', 'fabrication', 'assemblage', 'montage', 'usinage', 'machine outil',
        'reglage machine', 'controle qualite', 'norme iso', 'lean manufacturing', '5s', 'kaizen'
    ],
    'maintenance': [
        'maintenance', 'maintenance preventive', 'maintenance corrective', 'depannage',
        'reparation', 'diagnostic panne', 'entretien equipement', 'mecanique industrielle',
        'electrotechnique', 'automatisme', 'pneumatique', 'hydraulique'
    ],
    'qualite': [
        'qualite', 'controle qualite', 'assurance qualite', 'non conformite', 'audit qualite',
        'amelioration continue', 'qhse', 'hygiene securite', 'norme iso 9001'
    ],
    'electricite': [
        'electricite', 'electricien', 'installation electrique', 'cablage', 'tableau electrique',
        'habilitation electrique', 'maintenance electrique', 'courant faible', 'courant fort'
    ]
}

# Construction & BTP skills
skills_btp = {
    'construction': [
        'construction', 'batiment', 'travaux publics', 'gros oeuvre', 'second oeuvre',
        'maconnerie', 'coffrage', 'ferraillage', 'betonnage', 'chape', 'enduit'
    ],
    'metiers_btp': [
        'plomberie', 'plombier', 'chauffage', 'climatisation', 'menuiserie', 'menuisier',
        'charpentier', 'couverture', 'peinture', 'peintre', 'carrelage', 'carreleur'
    ],
    'conduite_travaux': [
        'conduite travaux', 'chef de chantier', 'gestion chantier', 'planning travaux',
        'lecture plan', 'mesurage', 'metre', 'suivi travaux', 'coordination chantier'
    ]
}

# Cuisine & Restauration skills
skills_cuisine_restauration = {
    'cuisine': [
        'cuisine', 'cuisinier', 'chef de cuisine', 'commis de cuisine', 'preparation culinaire',
        'mise en place', 'dressage', 'garde manger', 'patisserie', 'patissier', 'boulangerie',
        'respect hygiene', 'haccp', 'fiches techniques'
    ],
    'service': [
        'service', 'serveur', 'service table', 'prise commande', 'service boisson',
        'accueil client', 'restaurant', 'bar', 'barman', 'sommelier'
    ],
    'gestion_restauration': [
        'gestion stock alimentaire', 'inventaire', 'commande fournisseur', 'cout matiere',
        'menu engineering', 'rentabilite', 'ratio'
    ]
}

# Health & Medical skills
skills_sante = {
    'soins': [
        'soins', 'soins infirmiers', 'pansement', 'injection', 'perfusion', 'prelevement',
        'hygiene', 'desinfection', 'sterilisation', 'protocole soin', 'surveillance patient'
    ],
    'medical': [
        'medical', 'paramedicale', 'diagnostic', 'examen clinique', 'observation',
        'kinesitherapie', 'reeducation', 'radio', 'imagerie medicale', 'laboratoire'
    ],
    'accompagnement': [
        'accompagnement', 'aide personne', 'toilette', 'habillage', 'aide mobilite',
        'ecoute', 'bienveillance', 'patience', 'empathie'
    ]
}

# Agriculture & Environment skills
skills_agriculture = {
    'agriculture': [
        'agriculture', 'culture', 'plantation', 'semis', 'recolte', 'irrigation',
        'traitement phytosanitaire', 'agronomie', 'elevage', 'soin animaux', 'alimentation animaux'
    ],
    'environnement': [
        'environnement', 'developpement durable', 'gestion dechets', 'recyclage', 'compostage',
        'eau', 'assainissement', 'energie renouvelable'
    ],
    'materiel_agricole': [
        'tracteur', 'engin agricole', 'conduite engin', 'entretien materiel agricole'
    ]
}

# Security & Safety skills
skills_securite = {
    'securite': [
        'securite', 'surveillance', 'gardiennage', 'ronde', 'controle acces', 'videosurveillance',
        'intervention', 'alerte', 'main courante', 'sst', 'ssiap', 'incendie', 'evacuation'
    ]
}

# Education & Training skills
skills_education = {
    'enseignement': [
        'enseignement', 'pedagogie', 'cours', 'formation', 'formateur', 'animation formation',
        'programme pedagogique', 'evaluation', 'suivi apprenant', 'correction', 'tutoring'
    ]
}

# Support & Customer Service skills
skills_support = {
    'support_client': [
        'support', 'assistance', 'helpdesk', 'hotline', 'depannage', 'support technique',
        'support fonctionnel', 'sav', 'ticket', 'incident', 'resolution probleme',
        'teleconseiller', 'traitement appel', 'standard telephonique'
    ],
    'back_office': [
        'back office', 'saisie donnees', 'traitement dossier', 'gestion administrative',
        'suivi dossier', 'archivage', 'classement', 'verification', 'controle dossier'
    ]
}

# Soft skills - applicable to all domains
skills_soft = {
    'communication': [
        'communication', 'redaction', 'presentation', 'expression orale', 'expression ecrite',
        'ecoute active', 'diplomatie', 'persuasion', 'negociation', 'capacite redactionnelle'
    ],
    'organisation': [
        'organisation', 'planification', 'gestion temps', 'priorisation', 'rigueur', 'methode',
        'autonomie', 'proactivite', 'polyvalence', 'adaptabilite', 'flexibilite', 'reactivity'
    ],
    'relationnel': [
        'travail equipe', 'collaboration', 'esprit equipe', 'cooperation', 'relationnel',
        'empathie', 'sens service', 'bienveillance', 'patience', 'courtoisie'
    ],
    'analytical': [
        'analyse', 'resolution probleme', 'esprit analytique', 'pensee critique', 'synthese',
        'creativite', 'innovation', 'esprit initiative', 'curiosite', 'force proposition'
    ],
    'attitude': [
        'dynamisme', 'motivation', 'engagement', 'serieux', 'professionnalisme', 'ponctualite',
        'assiduite', 'disponibilite', 'sense responsabilite', 'integrite'
    ]
}

# Qualifications
qualifications = {
    'diplomes': [
        'bac', 'bac 2', 'bac 3', 'bac 4', 'bac 5', 'bac plus 2', 'bac plus 3', 'bac plus 5',
        'licence', 'licence professionnelle', 'master', 'master 1', 'master 2', 'doctorat',
        'ingenieur', 'diplome ingenieur', 'dut', 'bts', 'mba', 'diplome', 'cap', 'bep',
        'titre professionnel', 'diplome universitaire'
    ],
    'certifications': [
        'certification', 'certifie', 'certified', 'habilitation', 'agrement', 'caces',
        'sst', 'ssiap', 'haccp', 'pmp', 'itil', 'scrum master', 'product owner',
        'aws certified', 'azure certified', 'habilitation electrique', 'permis'
    ],
    'languages': [
        'francais', 'anglais', 'malagasy', 'malgache', 'espagnol', 'allemand', 'italien',
        'chinois', 'arabe', 'bilingue', 'trilingue', 'langue etrangere', 'niveau langue'
    ],
    'permis': [
        'permis b', 'permis c', 'permis d', 'permis be', 'permis ce', 'permis de conduire',
        'fimo', 'fcos', 'permis poids lourd', 'permis remorque'
    ]
}

# Compile all skills
all_skills_dict = {
    'informatique': skills_informatique,
    'finance_compta': skills_finance_compta,
    'commercial_vente': skills_commercial_vente,
    'rh_management': skills_rh_management,
    'supply_logistique': skills_supply_logistique,
    'marketing_communication': skills_marketing_communication,
    'industriel_technique': skills_industriel_technique,
    'btp': skills_btp,
    'cuisine_restauration': skills_cuisine_restauration,
    'sante': skills_sante,
    'agriculture': skills_agriculture,
    'securite': skills_securite,
    'education': skills_education,
    'support': skills_support,
    'soft_skills': skills_soft,
    'qualifications': qualifications
}

print(f"Skill dictionaries defined: {len(all_skills_dict)} main categories")


Skill dictionaries defined: 16 main categories


Functions to extract skills, experience, and qualifications from text

In [3]:
def extract_skills_from_text(text):
    """
    Extract skills from job offer text across all domains.
    Returns a dictionary with categorized skills found.
    """
    if not isinstance(text, str):
        return {}
    
    text_lower = text.lower()
    extracted_skills = {}
    
    # Iterate through all skill categories (all job domains)
    for domain, skills_categories in all_skills_dict.items():
        for category, skills_list in skills_categories.items():
            found_skills = []
            for skill in skills_list:
                # Use word boundaries to avoid partial matches
                pattern = r'\b' + re.escape(skill) + r'\b'
                if re.search(pattern, text_lower):
                    found_skills.append(skill)
            
            if found_skills:
                # Create unique key combining domain and category
                key = f"{domain}_{category}"
                extracted_skills[key] = found_skills
    
    return extracted_skills


def extract_experience_years(text):
    """
    Extract years of experience mentioned in the text.
    Returns the number of years or None.
    """
    if not isinstance(text, str):
        return None
    
    text_lower = text.lower()
    
    # Pattern to match experience in years
    patterns = [
        r'(\d+)\s*(?:ans?|annees?)\s*(?:d\s*)?(?:experience|exp)',
        r'(?:experience|exp)\s*(?:de\s*)?(\d+)\s*(?:ans?|annees?)',
        r'(\d+)\+\s*(?:ans?|annees?)',
        r'minimum\s*(\d+)\s*(?:ans?|annees?)',
    ]
    
    years_found = []
    for pattern in patterns:
        matches = re.findall(pattern, text_lower)
        for match in matches:
            try:
                years_found.append(int(match))
            except:
                pass
    
    if years_found:
        return min(years_found)
    return None


def get_all_skills_from_job(row):
    """
    Extract all skills from mission and profil columns combined.
    Returns a concatenated string of all found skills.
    """
    mission_text = row.get('mission_clean', '')
    profil_text = row.get('profil_clean', '')
    
    # Combine both texts
    combined_text = str(mission_text) + ' ' + str(profil_text)
    
    # Extract skills
    skills_dict = extract_skills_from_text(combined_text)
    
    # Flatten all skills into a list
    all_skills = []
    for category, skills_list in skills_dict.items():
        all_skills.extend(skills_list)
    
    # Remove duplicates and join
    unique_skills = list(set(all_skills))
    
    return ' '.join(unique_skills) if unique_skills else ''


def count_skills_per_category(row):
    """
    Count number of skills found per category.
    Returns a dictionary with counts.
    """
    mission_text = row.get('mission_clean', '')
    profil_text = row.get('profil_clean', '')
    
    combined_text = str(mission_text) + ' ' + str(profil_text)
    skills_dict = extract_skills_from_text(combined_text)
    
    counts = {}
    for category, skills_list in skills_dict.items():
        counts[category] = len(skills_list)
    
    return counts


print("Skill extraction functions defined successfully for all job categories")

Skill extraction functions defined successfully for all job categories


In [7]:
df.keys()

Index(['title_clean', 'mission_clean', 'cluster', 'famille_poste'], dtype='object')

Extract skills from all job offers in the dataset

In [6]:
# Apply skill extraction to dataframe

# Extract all skills as a concatenated string
df['skills_extracted'] = df.apply(get_all_skills_from_job, axis=1)

# Extract experience years
df['experience_years'] = df.apply(
    lambda row: extract_experience_years(str(row.get('mission_clean', '')) + ' ' + str(row.get('profil_clean', ''))), 
    axis=1
)

# Extract detailed skills by category
df['skills_detail'] = df.apply(
    lambda row: extract_skills_from_text(str(row.get('mission_clean', '')) + ' ' + str(row.get('profil_clean', ''))), 
    axis=1
)

# Count skills per category
df['skills_count'] = df.apply(count_skills_per_category, axis=1)

print(f"Skill extraction completed for {len(df)} job offers")

df.keys()

Skill extraction completed for 5935 job offers


Index(['title_clean', 'mission_clean', 'cluster', 'famille_poste',
       'skills_extracted', 'experience_years', 'skills_detail',
       'skills_count'],
      dtype='object')

Display examples of extracted skills

In [8]:
# Display sample of extracted skills

print("Sample of extracted skills:\n")
print("="*80)

# Show first few rows with skills
for idx in range(min(5, len(df))):
    row = df.iloc[idx]
    print(f"\nJob {idx + 1}:")
    print(f"Title: {row.get('title_clean', 'N/A')}")
    print(f"Cluster: {row.get('cluster', 'N/A')}")
    print(f"Skills extracted: {row['skills_extracted'][:200] if row['skills_extracted'] else 'None'}")
    print(f"Experience required: {row['experience_years']} years" if row['experience_years'] else "Experience required: Not specified")
    print("-"*80)

Sample of extracted skills:


Job 1:
Title: conseiller client
Cluster: 9
Skills extracted: service production couverture relation client
Experience required: Not specified
--------------------------------------------------------------------------------

Job 2:
Title: administration comptabilite
Cluster: 1
Skills extracted: cours comptabilite evaluation diplome excel
Experience required: Not specified
--------------------------------------------------------------------------------

Job 3:
Title: procurment officer
Cluster: 1
Skills extracted: service couverture disponibilite
Experience required: Not specified
--------------------------------------------------------------------------------

Job 4:
Title: coursier moto
Cluster: 1
Skills extracted: manutention
Experience required: Not specified
--------------------------------------------------------------------------------

Job 5:
Title: premiere seconde ligne
Cluster: 1
Skills extracted: ticket reseau
Experience required: Not specified
-

Identify the most common skills across all job offers

In [10]:
# Analyze top skills across all job offers

# Collect all skills into a list
all_skills_list = []
for skills_text in df['skills_extracted']:
    if skills_text:
        all_skills_list.extend(skills_text.split())

# Count frequency of each skill
skills_frequency = Counter(all_skills_list)

# Display top 20 most common skills
print("\nTop 20 most common skills across all job offers:")
print("="*80)

for skill, count in skills_frequency.most_common(20):
    percentage = (count / len(df)) * 100
    print(f"{skill:<30} | Count: {count:>4} | Found in {percentage:.1f}% of offers")


Top 20 most common skills across all job offers:
qualite                        | Count: 1359 | Found in 22.9% of offers
developpement                  | Count: 1193 | Found in 20.1% of offers
service                        | Count:  940 | Found in 15.8% of offers
communication                  | Count:  797 | Found in 13.4% of offers
securite                       | Count:  752 | Found in 12.7% of offers
reporting                      | Count:  687 | Found in 11.6% of offers
collaboration                  | Count:  668 | Found in 11.3% of offers
production                     | Count:  626 | Found in 10.5% of offers
maintenance                    | Count:  617 | Found in 10.4% of offers
client                         | Count:  577 | Found in 9.7% of offers
marketing                      | Count:  576 | Found in 9.7% of offers
formation                      | Count:  543 | Found in 9.1% of offers
vente                          | Count:  512 | Found in 8.6% of offers
analyse           

Analyze skill distribution across clusters

In [11]:
# Analyze skills by cluster

print("\nTop skills per cluster:")
print("="*80)

for cluster_id in sorted(df['cluster'].unique()):
    df_cluster = df[df['cluster'] == cluster_id]
    
    # Collect all skills for this cluster
    cluster_skills = []
    for skills_text in df_cluster['skills_extracted']:
        if skills_text:
            cluster_skills.extend(skills_text.split())
    
    if cluster_skills:
        cluster_skills_freq = Counter(cluster_skills)
        top_5_skills = cluster_skills_freq.most_common(5)
        
        print(f"\nCluster {cluster_id} ({len(df_cluster)} offers):")
        print(f"Top 5 skills: {', '.join([skill for skill, count in top_5_skills])}")
    else:
        print(f"\nCluster {cluster_id} ({len(df_cluster)} offers):")
        print(f"No skills extracted")


Top skills per cluster:

Cluster 0 (592 offers):
Top 5 skills: commercial, communication, developpement, client, marketing

Cluster 1 (4349 offers):
Top 5 skills: qualite, developpement, service, securite, maintenance

Cluster 2 (134 offers):
Top 5 skills: service, prospection, vente, francais, client

Cluster 3 (128 offers):
Top 5 skills: qualite, securite, service, production, supervision

Cluster 4 (51 offers):
Top 5 skills: paie, communication, gestion, recrutement, developpement

Cluster 5 (61 offers):
Top 5 skills: qualite, developpement, production, communication, collaboration

Cluster 6 (49 offers):
Top 5 skills: budgetaire, reporting, rentabilite, analyse, collaboration

Cluster 7 (43 offers):
Top 5 skills: communication, formation, qualite, reporting, culture

Cluster 8 (30 offers):
Top 5 skills: financier, tresorerie, comptabilite, comptable, reporting

Cluster 9 (25 offers):
Top 5 skills: client, service, supervision, relation, environnement

Cluster 10 (45 offers):
Top 5

Analyze years of experience requirements

In [12]:
# Analyze experience requirements

df_with_experience = df[df['experience_years'].notna()]

print("\nExperience Requirements Analysis:")
print("="*80)
print(f"Offers specifying experience: {len(df_with_experience)} / {len(df)} ({len(df_with_experience)/len(df)*100:.1f}%)")
print(f"Offers without experience requirement: {len(df) - len(df_with_experience)}")

if len(df_with_experience) > 0:
    print(f"\nExperience Statistics:")
    print(f"Minimum experience required: {df_with_experience['experience_years'].min():.0f} years")
    print(f"Maximum experience required: {df_with_experience['experience_years'].max():.0f} years")
    print(f"Average experience required: {df_with_experience['experience_years'].mean():.1f} years")
    print(f"Median experience required: {df_with_experience['experience_years'].median():.0f} years")
    
    print("\nExperience distribution:")
    experience_dist = df_with_experience['experience_years'].value_counts().sort_index()
    for years, count in experience_dist.items():
        print(f"{int(years)} years: {count} offers")


Experience Requirements Analysis:
Offers specifying experience: 0 / 5935 (0.0%)
Offers without experience requirement: 5935


Show dataframe with extracted skills columns

In [13]:
# Display dataframe with extracted skills columns

print("\nDataframe with skill extraction columns:")
print("="*80)

# Select relevant columns to display
columns_to_display = ['title_clean', 'cluster', 'skills_extracted', 'experience_years']
df_display = df[columns_to_display].copy()

df_display[0:20]


Dataframe with skill extraction columns:


,title_clean,cluster,skills_extracted,experience_years
0,conseiller client,9,service production couverture relation client,None
1,administration comptabilite,1,cours comptabilite evaluation diplome excel,None
2,procurment officer,1,service couverture disponibilite,None
3,coursier moto,1,manutention,None
4,premiere seconde ligne,1,ticket reseau,None
5,reseau,1,fidelisation developpement vente,None
6,exploitation,1,prise commande maintenance qualite livraison securite commande,None
7,charge commercial,0,fidelisation communication vente prospection reseaux sociaux reseau,None
8,conseillers clients anglophones,1,,None
9,charge commercial,0,formation assistance accompagnement,None


In [14]:
# Create comprehensive skill summary report

def get_category_skills(df, category_prefix):
    """Extract all skills from a specific category across all job offers"""
    all_category_skills = []
    for skills_dict in df['skills_detail']:
        if isinstance(skills_dict, dict):
            for key, skills_list in skills_dict.items():
                if category_prefix in key:
                    all_category_skills.extend(skills_list)
    return Counter(all_category_skills)

print("\n" + "="*80)
print("COMPREHENSIVE SKILL SUMMARY REPORT - ALL DOMAINS")
print("="*80)

# Define domains to report
domain_info = {
    'informatique': 'IT & INFORMATIQUE',
    'finance_compta': 'FINANCE & COMPTABILITE',
    'commercial_vente': 'COMMERCIAL & VENTE',
    'rh_management': 'RH & MANAGEMENT',
    'supply_logistique': 'SUPPLY CHAIN & LOGISTIQUE',
    'marketing_communication': 'MARKETING & COMMUNICATION',
    'industriel_technique': 'INDUSTRIEL & TECHNIQUE',
    'btp': 'BTP & CONSTRUCTION',
    'cuisine_restauration': 'CUISINE & RESTAURATION',
    'sante': 'SANTE & MEDICAL',
    'agriculture': 'AGRICULTURE & ENVIRONNEMENT',
    'securite': 'SECURITE',
    'education': 'EDUCATION & FORMATION',
    'support': 'SUPPORT & SERVICE CLIENT',
    'soft_skills': 'COMPETENCES TRANSVERSALES',
    'qualifications': 'QUALIFICATIONS & DIPLOMES'
}

# Report top skills for each domain
for domain_key, domain_title in domain_info.items():
    skills_counter = get_category_skills(df, domain_key)
    if skills_counter:
        print(f"\n{domain_title}:")
        print("-"*80)
        top_skills = skills_counter.most_common(8)
        for skill, count in top_skills:
            percentage = (count / len(df)) * 100
            print(f"  - {skill:<35} | {count:>4} offers ({percentage:>5.1f}%)")

print("\n" + "="*80)
print(f"Total job offers analyzed: {len(df)}")
print("="*80)


COMPREHENSIVE SKILL SUMMARY REPORT - ALL DOMAINS

IT & INFORMATIQUE:
--------------------------------------------------------------------------------
  - developpement                       | 1076 offers ( 18.1%)
  - reporting                           |  670 offers ( 11.3%)
  - reseau                              |  179 offers (  3.0%)
  - kpi                                 |  149 offers (  2.5%)
  - tableau                             |  123 offers (  2.1%)
  - excel                               |   65 offers (  1.1%)
  - sql                                 |   62 offers (  1.0%)
  - vue                                 |   55 offers (  0.9%)

FINANCE & COMPTABILITE:
--------------------------------------------------------------------------------
  - comptable                           |  419 offers (  7.1%)
  - comptabilite                        |  339 offers (  5.7%)
  - budget                              |  249 offers (  4.2%)
  - declarations fiscales               |  216 off

Aggregated view of skills per cluster

In [17]:
# Create summary dataframe for skill analysis per cluster

df_cluster_skills_summary = df.groupby('cluster').agg({
    'title_clean': 'count',
    'experience_years': ['mean', 'min', 'max'],
    'skills_extracted': lambda x: ' '.join(x.dropna())
}).round(1)

df_cluster_skills_summary.columns = ['offer_count', 'avg_experience', 'min_experience', 'max_experience', 'all_skills']

print("\nCluster Summary with Skills:")
print("="*80)
df_cluster_skills_summary[0:5]


Cluster Summary with Skills:


offer_count avg_experience min_experience max_experience  \
cluster                                                             
0                592            NaN            NaN            NaN   
1               4349            NaN            NaN            NaN   
2                134            NaN            NaN            NaN   
3                128            NaN            NaN            NaN   
4                 51            NaN            NaN            NaN   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     